# FCT 생성 노트북 페이지
--------------------------------
#### 자격증 / 학력 / 상벌 / 경력 / 병역 / 어학 / 발령
 - 시스템A 3사는 테이블이 각 회사마다 떨어져있기 때문에 brz(3사 각테이블) -> slv (3사 테이블 합치기) -> gld(slv에서 합친 한개의 테이블을 사용)
    - slv를 타는 이유 : 각 테이블마다 3번을 추가하게되면 향후 유지보수나 현재 쿼리 길이가 너무 길어져서 파악하기 난해하여 slv로 한개의 테이블 생성
 - 시스템B brz -> gld 수행

--------------------------------------

## FCT 자격증 f_hr_license
--------------------------------------
 - 시스템B + 시스템A 각각 유니온된 상태
    - 시스템B와 시스템A는 서로 각기 다른 로직을 사용하기 때문에 문제가 생기면 시스템B는 상위쿼리에서 시스템A는 하위 쿼리에서 각각 수정해야함
    - 시스템B를 기준으로 컬럼을 생성했기 때문에 시스템B엔 있는 컬럼이지만 시스템A에는 없을수도 있음 -> 그런 경우 시스템A는 NULL처리

 - 시스템A는 EMP_NO만 존재, EMP_ID는 존재하지않기 때문에 EMP_ID >  알파벳 + EMP_NO 으로 구성함 즉, EMP_ID와 EMP_NO는 같은 데이터

 - 대시보드에 추가하기 위해서 컬럼추가시, 시스템A & 시스템B 모두 추가해야 수정가능함.
   - UNION을 기준으로  쿼리 컬럼수와 순서가 일치하지않으면 테이블 수정 불가능
   - 컬럼수와 순서를 맞췄는데 오류가 나면 CAST도 확인해야함 -> 시스템B는 대부분 timestamp 형태, 시스템A는 text형태로 CAST 구문 추가해야함
   - 날짜형식은 바꾸면 데이터가 안나오는 경우가 많으므로, CAST 후, 데이터 나오는지 확인 필수!

In [ ]:
df = spark.sql("""
-- 시스템B 쿼리: 시스템별 자격증 정보를 추출하여 표준화된 스키마로 통합
SELECT
LICENSE.MOD_USER_ID  AS MOD_PRSN,  -- 최종 수정자 ID
LICENSE.MOD_DATE AS MOD_DTT,       -- 최종 수정 일시
LICENSE.TZ_CD AS TZ_CD,            -- 타임존 코드
LICENSE.TZ_DATE AS TZ_DTT,         -- 타임존 기준 일시
CAST(LICENSE.EMP_ID AS string ) AS EMP_ID,  -- 사번 데이터 타입 통일 (String)
LICENSE.PHM_LICENSE_ID AS EMP_LCNS_ID,      -- 자격증 고유 ID
LICENSE.LICENSE_COMPANY_NM AS ISSUE_ORG,    -- 발급 기관
LICENSE.LICENSE_TYPE_CD AS LCNS_DIV_CD,     -- 자격증 구분 코드
COMM_LICENSE_TY.CD_NM AS LCNS_DIV_NM,       -- 자격증 구분 명칭 (공통코드 결합)
COMM_LICENSE.CD_NM AS LCNS_NM,               -- 자격증 명칭 (공통코드 결합)
LICENSE.NOTE AS NOTE,                       -- 비고
LICENSE.BONUS_TYPE AS BONUS_PAY_DIV_CD,     -- 수당 지급 구분 코드
LICENSE.REG_YN AS REG_YN,                   -- 등록 여부
LICENSE.LICENSE_NO AS LCNS_NO,              -- 자격증 번호
to_timestamp(LICENSE.END_YMD , 'yyyyMMdd') AS VALID_DT,  -- 유효 종료일 Timestamp 변환
to_timestamp(LICENSE.STA_YMD , 'yyyyMMdd') AS GET_DT,    -- 자격 취득일 Timestamp 변환
LICENSE.PERSON_ID  AS PRSNL_ID,              -- 개인 식별 번호
LICENSE.LICENSE_CD AS LCNS_CD               -- 자격증 코드
FROM brz.system_b.PHM_LICENSE LICENSE
LEFT JOIN brz.system_b.common_code COMM_LICENSE ON COMM_LICENSE.CD_KIND ='PHM_LICENSE_CD' AND LICENSE.LICENSE_CD = COMM_LICENSE.CD
LEFT JOIN brz.system_b.common_code COMM_LICENSE_TY ON COMM_LICENSE_TY.CD_KIND ='PHM_LICENSE_TYPE_CD' AND LICENSE.LICENSE_TYPE_CD = COMM_LICENSE_TY.CD
  
UNION  -- 시스템A와 시스템B의 데이터 스키마를 일치시켜 통합 수행
-- 시스템A 쿼리: Silver 레이어(3사 통합 테이블)를 활용한 데이터 추출
SELECT 
NULL as MOD_PRSN,  -- 시스템A 미존재 컬럼 NULL 처리
l.UPDATE_DTS as MOD_DTT,  
NULL as TZ_CD,  
NULL as TZ_DTT,   
CASE  -- 시스템A의 사번 체계를 통합 EMP_ID 규격으로 변환 (Prefix 적용)
 WHEN l.COMPANY_CD = '3000' THEN 'a' || l.EMP_NO -- Company A 접두어 'a' 부여
 WHEN l.COMPANY_CD = '2000' THEN 'q' || l.EMP_NO -- Company B 접두어 'q' 부여
 WHEN l.COMPANY_CD = '1000' THEN 'f' || l.EMP_NO -- Company C 접두어 'f' 부여
  ELSE l.EMP_NO  
 END AS EMP_ID,
NULL as EMP_LCNS_ID,  
l.ISSUE_ISTN_NM as ISSUE_ORG,  
NULL as LCNS_DIV_CD,  
NULL as LCNS_DIV_NM,  
l.QUA_CARD_NM as LCNS_NM,  
l.RMK_DC as NOTE,  
NULL as BONUS_PAY_DIV_CD,  
NULL as REG_YN,  
NULL as LCNS_NO,  
to_timestamp(l.VLID_DT , 'yyyyMMdd') as VALID_DT,  -- 시스템B와 동일한 날짜 포맷팅 적용
to_timestamp(l.ACQS_DT , 'yyyyMMdd') as GET_DT,  
NULL as PRSNL_ID,  
NULL as LCNS_CD  
FROM slv.system_a.hr_license_mst l  -- 3개사 통합 뷰(Silver Layer) 참조
WHERE l.COMPANY_CD in ('1000', '2000', '3000') -- 유효 계열사 데이터 필터링
""")

In [ ]:
# gld 쌓기 
df.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_license");

In [ ]:
%%sql
-- gld에 쌓인 데이터 브릭스 테이블 안에 쌓기
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_license
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_license';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_license

## FCT 학력 정보  f_hr_scholar 
-----------------------------------------
 - 시스템B : 최종학력여부컬럼[LAST_YN]을 사용할수없음 (다수 회사에서 학력 여부를 표시하지않았기 때문에)
   - 최종학교명 로직 규칙 : 졸업일자가 존재하고, 졸업일자가 더 최신인 학교, 졸업일자가 동일한 경우, 학력코드가 더 높은쪽이 최종학교명으로 찍히게 설정함
 - (추가)시스템B : 전공명[MJR_NM]은 공통테이블에서 가져온 전공명으로 가져오되 , 만약  전공명이 null 인 경우에 ,  학력 팩트 자체적으로 담겨있는 전공명으로 사용하는걸로 대체함 - 24.10.23
   - 원래는 시스템B 공통코드에서 가져온 전공명을 사용하고 있었으나, 원천테이블에서 이분화 되는 데이터들이 존재함으로 위와 같이 대체되게 설정함

 - 시스템A : 최종학력컬럼 사용가능 [LAST_LEDU_YN = 'Y'] 일때 최종학교명으로 찍히게 설정함 
<br>

In [ ]:
DF = spark.sql("""
-- 시스템B 쿼리: 윈도우 함수를 활용한 최신/최고 학력 순위 부여
WITH ranked_scholar AS (  
        SELECT  
            SCH.PERSON_ID,  
            comm.CD_NM AS FIN_SCH_NM,  
            RANK() OVER (  
                PARTITION BY SCH.PERSON_ID  
                ORDER BY SCH.SCH_GRD_CD DESC, COALESCE(SCH.END_YM, '999912') DESC, SCH.STA_YM DESC, comm.CD_NM ASC  
            ) AS rn,  -- 학위 레벨 및 졸업일자 기준 최신 학력 순위 산출
            SCH.STA_YM AS STA_YM,  
            SCH.END_YM AS GRAD_DT,  
            SCH.SCH_KIND_CD AS SCH_KIND_CD,  
            SCH.SCH_GRD_CD AS SCH_GRD_CD  
        FROM brz.system_b.PHM_SCHOLAR SCH  
        LEFT JOIN brz.system_b.common_code comm ON comm.cd_kind = 'PHM_SCH_CD' AND sch.SCH_CD = comm.cd  
        WHERE SCH.END_YM IS NOT NULL 
    )  
    , final_scholar AS (  
        SELECT  
            PERSON_ID,  
            FIN_SCH_NM,  
            SCH_KIND_CD,  
            MAX(STA_YM) AS MAX_STA_YM,  
            MAX(GRAD_DT) AS MAX_GRAD_DT,  
            MAX(SCH_GRD_CD) AS MAX_SCH_GRD_CD  
        FROM ranked_scholar  
        WHERE rn = 1  -- 최상위 순위 데이터만 추출하여 최종 학력 특정
        GROUP BY PERSON_ID, FIN_SCH_NM, SCH_KIND_CD  
    )  
SELECT DISTINCT 
SCH.SUB_MAJOR_NM AS SUBMJR_NM,
SCH.SUB_MAJOR_CD AS SUBMJR_CD,
SCH.ENTRANCE_CD AS SCHENT_DIV_CD,
SCH.SCH_KIND_CD AS SCH_DIV_CD,
SCH.DAY_KIND_CD AS DAY_NIGHT_DIV_CD,
SCH.SCH_PLACE_CD AS SCH_PLC_CD,
SCH.SCH_PLACE_NM AS SCH_PLC_NM,
SCH.SCH_PLACE_DET_CD AS SCH_PLC_DTLS_CD,
SCH.SCH_PLACE_DET_NM AS PLC_REGION_DTLS,
SCH.LAST_YN AS LAST_SCHLST_YN,
SCH.CAREER_NUM AS CAREER_ADMIT_MTH_CNT,
SCH.NOTE AS NOTE,
SCH.MOD_USER_ID AS MOD_PRSN,
SCH.MOD_DATE AS MOD_DTT,
SCH.TZ_CD AS TZ_CD,
SCH.TZ_DATE AS TZ_DTT,
SCH.PHM_SCHOLAR_ID AS EMP_SCHLST_ID,
CAST(SCH.EMP_ID AS varchar(20)) AS EMP_ID,
SCH.PERSON_ID AS PRSNL_ID,
SCH.STA_YM AS SCHENT_YYYYMM,
SCH.END_YM AS GRAD_YYYYMM,
dsch.DTL_CD AS SCHLST_CD, -- 학력 등급 코드
SCH.SCH_CD AS SCH_CD,
cast(SCH.MAJOR_CD as VARCHAR(20)) AS MAJOR_CD,
SCH.DOU_MAJOR_CD AS DOU_MAJOR_CD,
SCH.DOU_MAJOR_NM AS DOU_MAJOR_NM,
dsch.DTL_NM   AS SCHLST_NM, 
cast(comm.CD_NM AS VARCHAR(40)) AS SCH_NM,
coalesce(list.CD_NM, SCH.MAJOR_NM) AS MJR_NM, -- 전공명 표준화: 공통 코드 우선 적용
CASE  
        WHEN comm.CD_NM = fs.FIN_SCH_NM AND (SCH.END_YM = fs.MAX_GRAD_DT OR SCH.END_YM IS NULL) AND SCH.STA_YM = fs.MAX_STA_YM AND SCH.SCH_GRD_CD = fs.MAX_SCH_GRD_CD AND (fs.MAX_SCH_GRD_CD >= '01' OR list.CD_NM IS NOT NULL)  
        THEN fs.FIN_SCH_NM  
        ELSE NULL  
    END AS LAST_SCH_NM -- 비즈니스 로직 기반 최종 학교명 매핑
FROM brz.system_b.PHM_SCHOLAR SCH   
LEFT JOIN brz.system_b.common_code comm ON comm.cd_kind = 'PHM_SCH_CD' AND sch.SCH_CD = comm.cd     
LEFT JOIN brz.system_b.common_code list  ON list.cd_kind = 'PHM_MAJOR_CD' AND sch.MAJOR_CD = list.cd 
LEFT JOIN ref.hr.code_master dsch ON dsch.df_cd = 'LaSchRatCl' AND sch.SCH_GRD_CD = dsch.Value1 
left join gld.hr.d_hr_school grad on sch.SCH_GRD_CD = grad.SCHL_CD  
LEFT JOIN final_scholar fs ON fs.PERSON_ID = SCH.PERSON_ID

UNION
-- 시스템A 쿼리: 시스템B 스키마와 일치화 및 사번 접두어 변환
SELECT 
NULL AS SUBMJR_NM,
NULL AS SUBMJR_CD,
NULL AS SCHENT_DIV_CD,
NULL AS SCH_DIV_CD,
NULL AS DAY_NIGHT_DIV_CD,
NULL AS SCH_PLC_CD,
NULL AS SCH_PLC_NM,
NULL AS SCH_PLC_DTLS_CD,
NULL AS PLC_REGION_DTLS,
sch.LAST_LEDU_YN as LAST_SCHLST_YN,
NULL AS CAREER_ADMIT_MTH_CNT,
NULL AS NOTE,
NULL AS MOD_PRSN,
sch.UPDATE_DTS as MOD_DTT,
NULL AS TZ_CD,
NULL AS TZ_DTT,
NULL AS EMP_SCHLST_ID,
CASE
WHEN sch.COMPANY_CD = '3000' THEN 'a' || sch.EMP_NO 
WHEN sch.COMPANY_CD = '2000' THEN 'q' || sch.EMP_NO 
WHEN sch.COMPANY_CD = '1000' THEN 'f' || sch.EMP_NO 
ELSE sch.EMP_NO
END AS EMP_ID, -- 회사별 접두어 구분을 통한 전사 통합 사번 생성
NULL AS PRSNL_ID,
TO_CHAR(TO_DATE(sch.MTRC_DT, 'yyyyMMdd'), 'yyyyMM') AS SCHENT_YYYYMM,  -- 입학일자 데이터 포맷팅
TO_CHAR(TO_DATE(sch.GRDT_DT, 'yyyyMMdd'), 'yyyyMM') AS GRAD_YYYYMM,    -- 졸업일자 데이터 포맷팅
COALESCE(dicd.DTL_CD, docd.DTL_CD, stcd.DTL_CD, NULL ) AS SCHLST_CD,
sch.SCHL_CD as SCH_CD,
cast (sch.DMJ_CD as VARCHAR(20)) as MAJOR_CD,
sch.PL_MJR_CD as DOU_MAJOR_CD,
sch.PL_MJR_NM as DOU_MAJOR_NM,
COALESCE(dicd.DTL_NM, docd.DTL_NM, stcd.DTL_NM, NULL ) AS SCHLST_NM, 
cast (sch.SCHL_NM  as VARCHAR(40))as SCH_NM,
sch.MJR_NM as MJR_NM,
CASE
WHEN sch.LAST_LEDU_YN = 'Y' THEN SCHL_NM -- 최종학력 여부('Y')에 따른 학교명 출력
ELSE NULL
END AS LAST_SCH_NM
from slv.system_a.HR_SCHOCARE_MST sch
LEFT JOIN (
    SELECT DISTINCT DTL_CD, DTL_NM, Value7 FROM ref.hr.code_master WHERE df_cd = 'LaSchRatCl' AND Value7 IS NOT NULL
) dicd ON dicd.Value7 = sch.LEDU_CD AND sch.COMPANY_CD = '3000' 
LEFT JOIN (
    SELECT DISTINCT DTL_CD, DTL_NM, Value3 FROM ref.hr.code_master WHERE df_cd = 'LaSchRatCl' AND Value3 IS NOT NULL
) docd ON docd.Value3 = sch.LEDU_CD AND sch.COMPANY_CD = '1000' 
LEFT JOIN (
    SELECT DISTINCT DTL_CD, DTL_NM, Value5 FROM ref.hr.code_master WHERE df_cd = 'LaSchRatCl' AND Value5 IS NOT NULL
) stcd ON stcd.Value5 = sch.LEDU_CD AND sch.COMPANY_CD = '2000' 
where COMPANY_CD in ('1000', '2000' , '3000')
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_scholar");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_scholar
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_scholar';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_scholar

## FCT 상벌정보 f_hr_reward_penalty
-----------------------------------------
 - 시스템B : 징계 & 포상이 한 테이블에 존재함 [brz.system_b.PPM_MNT]
 - 시스템A   : 징계 & 포상 각 다른 테이블로 존재함 [포상:SLV.system_a.hr_huprize_list / 징계:SLV.system_a.HR_HUDISCIP_LIST]  
 - 최종적으로 UNION이 3번 진행함 ( 시스템B 상벌 + 시스템A포상 + 시스템A징계)

In [ ]:
DF = spark.sql("""
-- 시스템B 쿼리
SELECT 
MNT.APPLY_YN AS CAM_APPLY_YN,
MNT.DEL_YN AS CAM_DEL_YN,
MNT.COMPANY_CD AS HR_FIELD_CD,
CAST(MNT.EMP_ID AS STRING) AS EMP_ID,
MNT.TYPE_CD AS RWD_RNTY_DIV_CD,
MNT.KIND_CD AS RWD_RNTY_KIND_CD,
MNT.INOFF_CD AS INOUT_HOUSE_DIV_CDD,
MNT.PPM_NO AS RWD_RNTY_NO,
cls.CD_NM AS RWD_RNTY_NM, --종류(상벌)
MNT.PPM_YMD AS RWD_RNTY_DT, --상벌일자
MNT.PPM_MON AS RWD_AMT,
MNT.PPM_DESC AS RWD_PNTY_RSN,
MNT.END_YMD AS END_DT,
MNT.CUST_COL1 AS PERCOM_DT_FRST,
MNT.INPUT_TYPE_CD AS INPUT_DATA_TYPE_CD,
MNT.MNT_POOL_ID AS MENTOR_POOL_ID,
MNT.CAM_YN AS CAM_REG_YN,
MNT.CAM_DATE AS CAM_LINK_DTT,
MNT.CAM_DOC_ID AS CAM_CNF_DOC_ID,
MNT.CAM_PRE_ID AS CAM_RCPNT_ID,
MNT.NOTE AS NOTE,
MNT.MOD_USER_ID AS MOD_PRSN,
MNT.MOD_DATE AS MOD_DTT,
MNT.TZ_CD AS TZ_CD,
MNT.TZ_DATE AS TZ_DTT,
MNT.PPM_MNT_ID AS RWD_DTLS_MNGMNT_ID
FROM brz.system_b.PPM_MNT MNT 
LEFT JOIN brz.system_b.common_code cls ON cls.CD_KIND = 'PPM_TYPE_CD' AND MNT.TYPE_CD= cls.CD

UNION
--시스템A 쿼리
-- 포상 
SELECT  
NULL AS CAM_APPLY_YN, 
NULL AS CAM_DEL_YN, 
NULL AS HR_FIELD_CD, 
CASE --사번  
    WHEN COMPANY_CD = '3000' THEN 'a' || EMP_NO -- Company A
    WHEN COMPANY_CD = '2000' THEN 'q' || EMP_NO -- Company B
    WHEN COMPANY_CD = '1000' THEN 'f' || EMP_NO -- Company C
    ELSE EMP_NO  
END AS EMP_ID,  
NULL AS RWD_RNTY_DIV_CD, 
NULL AS RWD_RNTY_KIND_CD, 
NULL AS INOUT_HOUSE_DIV_CDD,
NULL AS RWD_RNTY_NO, 
'포상' AS RWD_RNTY_NM, --포상명  
TO_TIMESTAMP(PRZ_DT  , 'yyyyMMdd') AS RWD_RNTY_DT, --포상일시  
NULL AS RWD_AMT, 
PRZ_DC AS RWD_PNTY_RSN, --포상내용  
NULL AS END_DT, 
NULL AS PERCOM_DT_FRST, 
NULL AS INPUT_DATA_TYPE_CD,  
NULL AS MENTOR_POOL_ID, 
NULL AS CAM_REG_YN,
NULL AS CAM_LINK_DTT, 
NULL AS CAM_CNF_DOC_ID, 
NULL AS CAM_RCPNT_ID,
NULL AS NOTE, 
NULL AS MOD_PRSN, 
UPDATE_DTS AS MOD_DTT,
NULL AS TZ_CD,
NULL AS TZ_DTT, 
NULL AS RWD_DTLS_MNGMNT_ID
FROM SLV.system_a.hr_huprize_list

UNION
-- 징계 
SELECT  
NULL AS CAM_APPLY_YN, 
NULL AS CAM_DEL_YN, 
NULL AS HR_FIELD_CD, 
CASE --사번  
    WHEN COMPANY_CD = '3000' THEN 'a' || EMP_NO -- Company A
    WHEN COMPANY_CD = '2000' THEN 'q' || EMP_NO -- Company B
    WHEN COMPANY_CD = '1000' THEN 'f' || EMP_NO -- Company C
    ELSE EMP_NO  
END AS EMP_ID,  
NULL AS RWD_RNTY_DIV_CD, 
NULL AS RWD_RNTY_KIND_CD, 
NULL AS INOUT_HOUSE_DIV_CDD,
NULL AS RWD_RNTY_NO, 
'징계' AS RWD_RNTY_NM, --포상명  
TO_TIMESTAMP(DCPL_DT , 'yyyyMMdd') AS RWD_RNTY_DT, --포상일시  
NULL AS RWD_AMT, 
DCPL_DC AS RWD_PNTY_RSN, --포상내용  
NULL AS END_DT, 
NULL AS PERCOM_DT_FRST, 
NULL AS INPUT_DATA_TYPE_CD,  
NULL AS MENTOR_POOL_ID, 
NULL AS CAM_REG_YN,
NULL AS CAM_LINK_DTT, 
NULL AS CAM_CNF_DOC_ID, 
NULL AS CAM_RCPNT_ID,
NULL AS NOTE, 
NULL AS MOD_PRSN, 
UPDATE_DTS AS MOD_DTT,
NULL AS TZ_CD,
NULL AS TZ_DTT, 
NULL AS RWD_DTLS_MNGMNT_ID
FROM SLV.system_a.HR_HUDISCIP_LIST;
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_reward_penalty");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_reward_penalty
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_reward_penalty';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_reward_penalty

## FCT 경력정보 f_hr_career
-----------------------------------------
 - 시스템B & 시스템A 모두 특별한 사항없음
<br>

In [ ]:
DF = spark.sql("""
SELECT  
-- 시스템B 쿼리 
    PHM_CAREER_ID AS EMP_CAREER_ID,  
    PLACE_NM AS PLC,  
    PERSON_ID AS PRSNL_ID,  
    STA_YMD AS WORK_STRT_DT,  
    END_YMD AS WORK_END_DT,  
    ORG_CORP_NM AS WORK_ORG_NM,  
    POSITION_NM AS LAST_JOB_GD_NM,  
    RETIRE_CAUSE AS RETRD_RSN,  
    RCAREER_NUM AS REAL_CAREER_MTH_CNT,  
    RECO_RATE AS ADMIT_RATE,  
    CAST(CAREER_NUM AS varchar(20)) AS ADMIT_CAREER_MTH_CNT,  
    NOTE AS NOTE,  
    MOD_USER_ID AS MOD_PRSN,  
    MOD_DATE AS MOD_DTT,  
    TZ_CD AS TZ_CD,  
    TZ_DATE AS TZ_DTT,  
    ORG_ENG_NM AS EMGL_CMP_NM,  
    CAST(EMP_ID AS varchar(20)) AS EMP_ID,
    WORK_NM 
FROM  brz.system_b.phm_career  
  
UNION  
  --시스템A 쿼리
SELECT  
    NULL AS EMP_CAREER_ID,  
    NULL AS PLC,  
    NULL AS PRSNL_ID,  
   to_timestamp(JNCO_DT, 'yyyyMMdd') AS WORK_STRT_DT,  
   to_timestamp(RETR_DT, 'yyyyMMdd') AS WORK_END_DT,  
    COMPANY_NM AS WORK_ORG_NM,  
    NULL AS LAST_JOB_GD_NM,  
    NULL AS RETRD_RSN,  
    NULL AS REAL_CAREER_MTH_CNT,  
    NULL AS ADMIT_RATE,  
   CAST(CAST(ROUND(MONTHS_BETWEEN(to_date(RETR_DT, 'yyyyMMdd'), to_date(JNCO_DT, 'yyyyMMdd')), 0) AS INT) AS VARCHAR(20)) AS ADMIT_CAREER_MTH_CNT,
    NULL AS NOTE,  
    NULL AS MOD_PRSN,  
    UPDATE_DTS as MOD_DTT,  
    NULL AS TZ_CD,  
    NULL AS TZ_DTT,  
    NULL AS EMGL_CMP_NM,  
    CASE  
        WHEN COMPANY_CD = '3000' THEN 'a' || EMP_NO -- Company A
        WHEN COMPANY_CD = '2000' THEN 'q' || EMP_NO -- Company B
        WHEN COMPANY_CD = '1000' THEN 'f' || EMP_NO -- Company C
        ELSE EMP_NO  
    END AS EMP_ID,
    RSPT_TSK_NM
FROM  slv.system_a.hr_career_mst  
WHERE COMPANY_CD in ('1000', '2000', '3000')
""")

In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY") # Spark 3.0부터 날짜 파싱이 변경되어서 오류가 발생하는 경우가 있어서 설정

DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_career");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_career
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_career';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_career

## FCT 병역정보 f_hr_military
-----------------------------------------
 - 시스템B & 시스템A 모두 특별한 사항없음
<br>

In [ ]:
DF = spark.sql("""
 --시스템B 쿼리
SELECT 
 cast(ARMY.EMP_ID as VARCHAR(20)) AS EMP_ID
,ARMY.ARMY_NO_CD AS INCMPL_RSN
,ARMY.ARMY_SERV_CD AS SRV_TYPE_CD
,ARMY.PHM_ARMY_ID AS EMP_MIL_SRVC_ID
,ARMY.ARMY_NO_REASON_CD AS MIL_CPLT_DIV_CD
,ARMY.ARMY_TYPE_CD AS MIL_TYPE_CD
,typ.CD_NM AS MIL_TYPE_NM --군별명
,ARMY.ARMY_BRANCH_CD AS MIL_BRNCH_CD
,ARMY.ARMY_CLASS_CD AS MIL_RANK_CD
,cla.CD_NM AS MIL_RANK_NM -- 계급명
,ARMY.ARMY_NO AS MIL_NO
,ARMY.ARMY_DISCHARGE_CD AS DSCHRG_DIV_CD
,ARMY.IN_YMD AS MIL_JOIN_DT
,ARMY.OUT_YMD AS DSCHRG_DT
,ARMY.PLUS_MM AS WRKPLC_RSRV_ARMY_JOIN_YN
,ARMY.ARMY_CLASSIFICATION_CD AS MIL_CLASS_CD
,cls.CD_NM AS MIL_CLASS_NM --역종명
,ARMY.ARMY_MTALENT_CD AS MIL_SP_CD
,mtl.CD_NM AS MIL_SP_NM --주특기명
,ARMY.NOTE AS NOTE
,ARMY.MOD_USER_ID AS MOD_PRSN
,ARMY.MOD_DATE AS MOD_DTT
,ARMY.TZ_CD AS TZ_CD
,ARMY.TZ_DATE AS TZ_DTT
,ARMY.PERSON_ID AS PRSNL_ID
,ROUND(MONTHS_BETWEEN(TO_DATE(DSCHRG_DT, 'yyyy-MM-dd'), TO_DATE(MIL_JOIN_DT, 'yyyy-MM-dd'))) AS MIL_ADMIT_MTH_CNT
FROM brz.system_b.PHM_ARMY ARMY
LEFT JOIN brz.system_b.common_code cls ON cls.CD_KIND ='PHM_ARMY_CLASSIFICATION_CD' and army.ARMY_CLASSIFICATION_CD = cls.cd --역종
LEFT JOIN brz.system_b.common_code mtl ON mtl.CD_KIND ='PHM_ARMY_MTALENT_CD' and army.ARMY_MTALENT_CD = mtl.cd --주특기명
LEFT JOIN brz.system_b.common_code typ ON typ.CD_KIND ='PHM_ARMY_TYPE_CD' and army.ARMY_TYPE_CD = typ.cd --군별
LEFT JOIN brz.system_b.common_code cla ON cla.CD_KIND ='PHM_ARMY_CLASS_CD' and army.ARMY_CLASS_CD = cla.cd --계급


union 
-- 시스템A 쿼리 
SELECT DISTINCT
    CASE
        WHEN f.COMPANY_CD = '3000' THEN 'a' || f.EMP_NO -- Company A
        WHEN f.COMPANY_CD = '2000' THEN 'q' || f.EMP_NO -- Company B
        WHEN f.COMPANY_CD = '1000' THEN 'f' || f.EMP_NO -- Company C
        ELSE f.EMP_NO
    END AS EMP_ID, -- 사원ID
    f.EXMT_REASON_CD AS INCMPL_RSN,
    null as SRV_TYPE_CD,
    null as EMP_MIL_SRVC_ID,
    null as MIL_CPLT_DIV_CD,
    f.AMFT_CD AS MIL_TYPE_CD, -- 군별
    type.SYSDEF_NM AS MIL_TYPE_NM, -- 공통코드 테이블 / HR/P00770
    null as MIL_BRNCH_CD, 
    f.MLH_CD AS MIL_RANK_CD, -- 계급코드
    rank.SYSDEF_NM AS MIL_RANK_NM, -- 공통코드 테이블 / HR/P00800
    null as MIL_NO, 
    null as DSCHRG_DIV_CD, 
    TO_TIMESTAMP(f.ENST_DT, 'yyyyMMdd') AS MIL_JOIN_DT, -- 입대일자
    TO_TIMESTAMP(f.LAYOFF_DT, 'yyyyMMdd') AS DSCHRG_DT, -- 전역일자
    NULL AS WRKPLC_RSRV_ARMY_JOIN_YN, 
    f.COSS_CD AS MIL_CLASS_CD, -- 역종코드
    clas.SYSDEF_NM AS MIL_CLASS_NM, -- 공통코드 테이블 / HR/P06990
    NULL AS MIL_SP_CD, 
    NULL AS MIL_SP_NM, 
    NULL AS NOTE, 
    NULL AS MOD_PRSN, 
    f.UPDATE_DTS AS MOD_DTT, 
    NULL AS TZ_CD, 
    NULL AS TZ_DTT, 
    NULL AS PRSNL_ID, 
    ROUND(MONTHS_BETWEEN(to_timestamp(f.LAYOFF_DT, 'yyyyMMdd'), to_timestamp(f.ENST_DT, 'yyyyMMdd'))) AS MIL_ADMIT_MTH_CNT
FROM slv.system_a.HR_EMPINFO_DTL f 
LEFT JOIN slv.system_a.ma_codedtl type ON type.SYSDEF_CD = f.AMFT_CD AND type.MODULE_CD = 'HR' AND type.FIELD_CD ='P00770' AND type.COMPANY_CD IN ('1000', '2000', '3000')
LEFT JOIN slv.system_a.ma_codedtl rank ON rank.SYSDEF_CD = f.AMFT_CD AND rank.MODULE_CD = 'HR' AND rank.FIELD_CD ='P00800' AND rank.COMPANY_CD IN ('1000', '2000', '3000')
LEFT JOIN slv.system_a.ma_codedtl clas ON clas.SYSDEF_CD = f.AMFT_CD AND clas.MODULE_CD = 'HR' AND clas.FIELD_CD ='P06990' AND clas.COMPANY_CD IN ('1000', '2000', '3000')
WHERE f.company_cd IN ('1000', '2000', '3000'); 
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_military");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_military
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_military';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_military

## FCT 어학정보 f_hr_language
-----------------------------------------
 - 시스템B & 시스템A 모두 특별한 사항없음
<br>

In [ ]:
DF = spark.sql("""
--시스템B 쿼리
SELECT 
est.PHM_LANG_EST_ID AS EMP_LANG_ESTM_ID -- 사원어학평가ID 
,CAST(est.EMP_ID as varchar(20)) AS EMP_ID -- 사원ID 
,est.PERSON_ID AS PRSNL_ID -- 개인ID 
,est.LANG_CD AS LANG_CD -- 어학코드 [PHM_LANG_CD] 
,LANG.cd_nm  AS LANG_NM
,est.STD_YY AS STD_YYYY -- 년도 
,est.EST_YMD AS ESTM_DT-- 평가일자 
,est.VALID_YMD AS VALID_DT -- 유효일자 
,est.EST_CD AS LANG_DIV_CD -- 어학구분코드[PHM_EST_CD] 
,estc.cd_nm AS LANG_DIV_NM --어학구분명
,est.EST_ORG_CD AS ESTM_ORG_CD -- 평가기관코드 [PHM_EST_ORG_CD] 
,ORG.cd_nm AS EST_ORG_NM --평가기관명
,est.EST_PNT AS LANG_SCR -- 어학점수, Functional level 
,est.EST_GRD_CD AS LANG_GRD_CD -- 어학등급코드 [PHM_EST_GRD_CD] 
,GRD.cd_nm AS LANG_GRD_NM --어학등급명
,est.INTERNAL_PNT AS PRSNT_LEV -- Present Level 
,est.NOTE -- 비고
,est.MOD_USER_ID AS MOD_PRSN
,est.MOD_DATE AS MOD_DTT
,est.TZ_CD AS TZ_CD
,est.TZ_DATE AS TZ_DTT
FROM brz.system_b.PHM_LANG_EST est 
LEFT JOIN brz.system_b.common_code LANG ON LANG.CD_KIND ='PHM_LANG_CD' and est.LANG_CD = LANG.cd --어학코드 
LEFT JOIN brz.system_b.common_code estc ON estc.CD_KIND ='PHM_EST_CD' and est.EST_CD = estc.cd --어학구분코드 [PHM_EST_CD] 
LEFT JOIN brz.system_b.common_code ORG ON ORG.CD_KIND ='PHM_EST_ORG_CD' and est.EST_ORG_CD = ORG.cd --평가기관코드 [PHM_EST_ORG_CD] 
LEFT JOIN brz.system_b.common_code GRD ON GRD.CD_KIND ='PHM_EST_GRD_CD' and est.EST_GRD_CD = GRD.cd --어학등급코드 [PHM_EST_GRD_CD] 

union 

SELECT 
--시스템A 쿼리
    NULL AS EMP_LANG_ESTM_ID, 
    CASE
        WHEN f.COMPANY_CD = '3000' THEN 'a' || f.EMP_NO -- Company A
        WHEN f.COMPANY_CD = '2000' THEN 'q' || f.EMP_NO -- Company B
        WHEN f.COMPANY_CD = '1000' THEN 'f' || f.EMP_NO -- Company C
        ELSE f.EMP_NO
    END AS EMP_ID, -- 사원ID
    NULL AS PRSNL_ID,
    f.LANG_CD, -- 어학코드
    m.SYSDEF_NM AS LANG_NM, -- 어학명
    NULL AS STD_YYYY,
    TO_timestamp(f.APYEXM_DT, 'yyyyMMdd') AS ESTM_DT, -- 평가일자
    TO_timestamp(f.VLID_DT ,'yyyyMMdd') AS VALID_DT, -- 유효일자
    f.LANG_FG_CD AS LANG_DIV_CD, -- 어학구분코드
    fg.SYSDEF_NM AS LANG_DIV_NM, -- 어학구분명
    f.TRIAL_ISTN_CD AS ESTM_ORG_CD, -- 평가기관코드
    f.TRIAL_ISTN_NM AS EST_ORG_NM, -- 평가기관명
    f.LANG_PT AS LANG_SCR, -- 어학점수
    f.LANG_GRD_CD AS LANG_GRD_CD, -- 어학등급코드
    f.LANG_GRD_NM AS LANG_GRD_NM, -- 어학등급명
    NULL AS PRSNT_LEV,
    f.RMK_DC AS NOTE, 
    NULL AS MOD_PRSN,
    f.UPDATE_DTS AS MOD_DTT,
    NULL AS TZ_CD, 
    NULL AS TZ_DTT 
FROM slv.system_a.hr_forelang_mst f 
LEFT JOIN slv.system_a.ma_codedtl m ON m.SYSDEF_CD = f.lang_cd AND m.MODULE_CD = 'HR' AND m.FIELD_CD ='Z001_20343' AND m.COMPANY_CD IN ('1000', '2000', '3000')
LEFT JOIN slv.system_a.ma_codedtl fg ON fg.SYSDEF_CD = f.lang_cd AND fg.MODULE_CD = 'HR' AND fg.FIELD_CD ='P00930' AND fg.COMPANY_CD IN ('1000', '2000', '3000')
where f.COMPANY_CD in ( '1000' , '2000' , '3000');
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_language");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_language
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_language';

In [ ]:
%%sql
select count(*) from gld.hr.f_hr_language

## FCT 발령정보 f_hr_appoint_history
--------------------------
- 발령정보 = 시스템B발령 + 시스템B수기발령 + 시스템A발령  총 3가지 테이블 유니온
- 시스템B발령 : brz.system_b.CAM_HISTORY  /  시스템B 수기발령 : BRZ.system_b.cam_history_bef  /  시스템A 발령 : slv.system_a.cam_past
- 시스템A : 해당 과거 발령 slv.system_a.cam_past 테이블은 brz단에서 쿼리로 처리해서 가져옴
- 시스템A :  직급 컬럼은 ** POSI_CD[직위]**로!!  /   _!![PSTN_CD 사용안함]!! _
- 자발 / 비자발 : 시스템A는 퇴직인 경우에 전체 자발/비자발 표현X ( 자발/비자발 마스터 : HR_EMP_SDTL 테이블에 존재하는 EMP_NO만 구분가능 그 외 구분불가)

In [ ]:
DF = spark.sql("""
SELECT DISTINCT  --시스템B 발령
cam.REN_YMD AS LAYOFF_RTRN_SCHDL_DT
,cam.DIS_YN AS DSPTCH_YN
,cam.DIS_KIND_CD AS DSPTCH_DIV_CD
,cam.DIS_RTN_YMD AS DSPTCH_RTRN_SCHDL_DT
,cam.DIS_ORG_ID AS DSPTCH_DEPT_CD
,cam.DIS_AREA_NM AS DSPTCH_PLC_NM
,cam.HIRE_CD AS HIRED_DIV_CD
,cam.PUNISH_END_YMD AS PENAL_CNCL_SCHDL_DT
,cam.MAS_YN AS HR_RFLCTN_YN
,cam.PAY_DUTY_CD AS SLRY_JOB_TTLE_GRD
,cam.PRINT_YN AS PRNT_YN
,cam.CUST_COL2 AS JOB_GRP_CD
,cam.CUST_COL3 AS RFRNC_DIV_CD
,cam.CUST_COL4 AS ENTRST_YN
,cam.CUST_COL5 AS ENTRST_END_SCHDL_DT
,cam.CUST_COL7 AS DEL_YN
,cam.CUST_COL8 AS CNF_DOC_NO
,cam.CUST_COL9 AS CNF_DOC_NM
,cam.CUST_COL10 AS CAM_RSN
,cam.NOTE AS NOTE
,cam.MOD_USER_ID AS MOD_PRSN
,cam.MOD_DATE AS MOD_DTT
,cam.TZ_CD AS TZ_CD
,cam.TZ_DATE AS TZ_DTT
,cam.SUB_COMPANY_CD AS CMP_DIV_CD --회사코드
,com.DTL_NM AS CMP_DIV_NM -- 회사명
,cam.PRO_GRADE_CD AS SLRY_STRT_SLRY_GRD_CD
,cam.PRO_KIND_CD AS FIRST_APPT_SLRY_KIND_CD
,cam.CNT_AMT AS YRLY_SLRY_AMT
,cam.SEND_YN AS JOB_SCND_YN
,cam.SEND_SUB_COMPANY_CD AS JOB_SCND_CAM_CMP_CD
,cam.SEND_JOP_CD AS JOB_SCND_JOB_GRP_CD
,cam.RETIRE_TYPE_CD AS RETRD_RSN_CD
,retire.CD_NM AS RETRD_RSN_NM
,cam.PAY_GRADE AS SLRY_GRD
,cam.PAY_JOP_CD AS SLRY_JOB_GRP
,cam.NEXT_POS_YMD AS NEXTERM_JOB_GD_PRMT_DT
,cam.CAM_HISTORY_ID AS CAM_HIST_ID
,CAST(cam.EMP_ID AS STRING) AS EMP_ID
,cam.PERSON_ID AS PRSNL_ID
,cam.CAM_DOC_ID AS CAM_CNF_ID
,cam.STA_YMD AS CAM_STRT_DT
,cam.END_YMD AS CAM_END_DT
,cam.CAM_YMD AS CAM_DT --발령일자
,cam.SEQ AS CAM_ORDER
,cam.TYPE_CD AS CAM_CD
,COALESCE(mre.dtl_cd ,cam.CAU_CD , null) AS CAM_CLASS_CD
,cam.COMPANY_CD AS CAM_HR_ZONE_CD
,cam.ORG_ID AS CAM_DEPT_ID
,cam.POS_CD as CAM_JOB_RANK_CD --직급코드 
,rank.CD_NM as CAM_JOB_RANK_NM --직급명
,cam.JOB_CD AS JOB_CD
,cam.DUTY_CD as CAM_JOB_TTLE_CD --직책코드 
,duty.cd_nm as CAM_JOB_TTLE_NM --직책명
,cam.EMP_KIND_CD AS CAM_EMP_KIND_CD
,cam.PAY_KIND_CD AS CAM_YRLY_SLRY_TYPE
,cam.PROBATION_YN AS PRBTN_YN
,cam.PROBATION_END_YMD AS PRBTN_END_DT
,cam.SEND_ORG_ID AS JOB_SCND_DEPT_ID
,cam.SEND_POS_CD AS JOB_SCND_JOB_GD_CD
,cam.SEND_POS_GRD_CD AS JOB_SCND_JOB_RANK_CD
,cam.SEND_JOB_CD AS JOB_SCND_JOB_CD
,cam.SEND_DUTY_CD AS JOB_SCND_JOB_TTLE_CD
,cam.REN_YN AS LAYOFF_YN
,COALESCE(mre.dtl_nm ,cls.CD_NM , null) AS CAM_CLASS_NM
 ,COALESCE(mre.dtl_cd ,'EE9999') AS RETRD_TYPE
  ,case when COALESCE(mre.dtl_nm , '코드없음') = '퇴직(자발적)' then '자발적'
       when COALESCE(mre.dtl_nm ,'코드없음') = '퇴직(비자발적)' then '비자발적'
  else null
  end RETRD_TYPE_NM
FROM brz.system_b.CAM_HISTORY cam 
LEFT JOIN brz.system_b.common_code cls ON cls.CD_KIND ='CAM_CAU_CD' and cam.CAU_CD = cls.cd --발령분류코드
left join ref.hr.code_master mre ON mre.df_cd = 'VolRetCl' AND cam.CAU_CD = mre.Value1 --공통코드에서 발령구분
LEFT JOIN brz.system_b.common_code retire on retire.CD_KIND ='CAM_RETIRE_CAU_CD' and cam.RETIRE_TYPE_CD = retire.CD
left join common_schema.apps.m_com_cm_cd_mn com on com.DF_CD = 'ComCl' and cam.SUB_COMPANY_CD = com.Value3 -- 회사명
LEFT JOIN brz.system_b.common_code duty on duty.CD_KIND ='PHM_DUTY_CD' and cam.DUTY_CD = duty.CD --직책코드 (공통 X)
LEFT JOIN brz.system_b.common_code rank on rank.CD_KIND ='PHM_POS_CD' and cam.POS_CD = rank.CD --직급코드(공통X)


UNION ALL
--시스템B 수기 발령부분 
SELECT DISTINCT  
NULL AS LAYOFF_RTRN_SCHDL_DT,  
NULL AS DSPTCH_YN,  
NULL AS DSPTCH_DIV_CD,  
NULL AS DSPTCH_RTRN_SCHDL_DT,  
NULL AS DSPTCH_DEPT_CD,  
NULL AS DSPTCH_PLC_NM,  
NULL AS HIRED_DIV_CD,  
NULL AS PENAL_CNCL_SCHDL_DT,  
NULL AS HR_RFLCTN_YN,  
NULL AS SLRY_JOB_TTLE_GRD,  
NULL AS PRNT_YN,  
NULL AS JOB_GRP_CD,  
NULL AS RFRNC_DIV_CD,  
NULL AS ENTRST_YN,  
NULL AS ENTRST_END_SCHDL_DT,  
NULL AS DEL_YN,  
NULL AS CNF_DOC_NO,  
NULL AS CNF_DOC_NM,  
NULL AS CAM_RSN,  
BEF.note AS NOTE,  
BEF.mod_user_id AS MOD_PRSN,  
BEF.mod_date AS MOD_DTT,  
BEF.tz_cd AS TZ_CD,  
BEF.tz_date AS TZ_DTT,  
null as CMP_DIV_CD,
BEF.sub_company_nm AS CMP_DIV_NM,  
NULL AS SLRY_STRT_SLRY_GRD_CD,  
NULL AS FIRST_APPT_SLRY_KIND_CD,  
NULL AS YRLY_SLRY_AMT,  
NULL AS JOB_SCND_YN,  
NULL AS JOB_SCND_CAM_CMP_CD,  
NULL AS JOB_SCND_JOB_GRP_CD,  
NULL AS RETRD_RSN_CD,  
NULL AS RETRD_RSN_NM,  
NULL AS SLRY_GRD,  
NULL AS SLRY_JOB_GRP,  
NULL AS NEXTERM_JOB_GD_PRMT_DT,  
NULL AS CAM_HIST_ID,  
cast(BEF.emp_id AS STRING) AS EMP_ID,  
NULL AS PRSNL_ID,  
NULL AS CAM_CNF_ID,  
NULL AS CAM_STRT_DT,  
NULL AS CAM_END_DT,  
BEF.sta_ymd AS CAM_DT,  
NULL AS CAM_ORDER,  
NULL AS CAM_CD,  
COALESCE(mre.dtl_cd ,BEF.CAU_CD , null) AS CAM_CLASS_CD,
NULL AS CAM_HR_ZONE_CD,  
NULL AS CAM_DEPT_ID,  
NULL AS CAM_JOB_RANK_CD,  
BEF.pos_nm AS CAM_JOB_RANK_NM,  
NULL AS JOB_CD,  
NULL AS CAM_JOB_TTLE_CD,  
BEF.duty_nm AS CAM_JOB_TTLE_NM,  
NULL AS CAM_EMP_KIND_CD,  
NULL AS CAM_YRLY_SLRY_TYPE,  
NULL AS PRBTN_YN,  
NULL AS PRBTN_END_DT,  
NULL AS JOB_SCND_DEPT_ID,  
NULL AS JOB_SCND_JOB_GD_CD,  
NULL AS JOB_SCND_JOB_RANK_CD,  
NULL AS JOB_SCND_JOB_CD,  
NULL AS JOB_SCND_JOB_TTLE_CD,  
NULL AS LAYOFF_YN,  
COALESCE(mre.dtl_nm ,cls.CD_NM , null) AS CAM_CLASS_NM,
COALESCE(mre.dtl_cd , 'EE9999') AS RETRD_TYPE,
case when COALESCE(mre.dtl_nm , null) = '퇴직(자발적)' then '자발적'
      when COALESCE(mre.dtl_nm , null) = '퇴직(비자발적)' then '비자발적'
  else '코드없음'  end RETRD_TYPE_NM
FROM BRZ.system_b.cam_history_bef BEF  
LEFT JOIN brz.system_b.common_code cls ON cls.CD_KIND ='CAM_CAU_CD' and BEF.CAU_CD = cls.cd --발령분류코드
left join ref.hr.code_master mre ON mre.df_cd = 'VolRetCl' AND BEF.CAU_CD = mre.Value1 --공통코드에서 발령구분

union all -- 시스템A 발령

SELECT
NULL AS LAYOFF_RTRN_SCHDL_DT,   
NULL AS DSPTCH_YN,   
NULL AS DSPTCH_DIV_CD, 
NULL AS DSPTCH_RTRN_SCHDL_DT, 
NULL AS DSPTCH_DEPT_CD,  
NULL AS DSPTCH_PLC_NM, 
NULL AS HIRED_DIV_CD,  
NULL AS PENAL_CNCL_SCHDL_DT,  
NULL AS HR_RFLCTN_YN,  
NULL AS SLRY_JOB_TTLE_GRD,  
NULL AS PRNT_YN,  
NULL AS JOB_GRP_CD,  
NULL AS RFRNC_DIV_CD,   
NULL AS ENTRST_YN,   
NULL AS ENTRST_END_SCHDL_DT,   
NULL AS DEL_YN,   
NULL AS CNF_DOC_NO,  
NULL AS CNF_DOC_NM, 
NULL AS CAM_RSN,
past.RMK_DC as NOTE, --비고
NULL AS MOD_PRSN,
NULL AS MOD_DTT,
NULL AS TZ_CD,   
NULL AS TZ_DTT, 
case when past.COMPANY_CD = '1000' then '1f'
     when past.COMPANY_CD = '2000' then '1q'
     when past.COMPANY_CD = '3000' then '1a'
 else past.COMPANY_CD
    END CMP_DIV_CD, -- 회사구분코드 
case when past.COMPANY_CD = '2000' then BIZAREA_NM
  else BIZAREA_NM  end  CMP_DIV_NM, -- 회사구분명
NULL AS SLRY_STRT_SLRY_GRD_CD,  
NULL AS FIRST_APPT_SLRY_KIND_CD,   
NULL AS YRLY_SLRY_AMT,  
NULL AS JOB_SCND_YN, 
NULL AS JOB_SCND_CAM_CMP_CD, 
NULL AS JOB_SCND_JOB_GRP_CD,  
NULL AS RETRD_RSN_CD, 
NULL AS RETRD_RSN_NM,  
NULL AS SLRY_GRD, 
NULL AS SLRY_JOB_GRP,  
NULL AS NEXTERM_JOB_GD_PRMT_DT,  
NULL AS CAM_HIST_ID,
CASE    
    WHEN past.COMPANY_CD = '3000' THEN 'a' || past.EMP_NO -- Company A
    WHEN past.COMPANY_CD = '2000' THEN 'q' || past.EMP_NO -- Company B
    WHEN past.COMPANY_CD = '1000' THEN 'f' || past.EMP_NO -- Company C
    ELSE past.EMP_NO    
END AS EMP_ID, --사원
NULL AS PRSNL_ID,  
NULL AS CAM_CNF_ID,  
NULL AS CAM_STRT_DT,   
NULL AS CAM_END_DT,
to_timestamp(past.GNFD_DT , 'yyyyMMdd') AS CAM_DT, --발령일자 
NULL AS CAM_ORDER,   
NULL AS CAM_CD,
NULL AS CAM_CLASS_CD,
NULL AS CAM_HR_ZONE_CD,   
NULL AS CAM_DEPT_ID,  
NULL AS CAM_JOB_RANK_CD,
past.POSI_NM as CAM_JOB_RANK_NM,  -- 직급명
NULL AS CAM_JOB_TTLE_CD, 
NULL AS JOB_CD, 
past.ODTY_NM as CAM_JOB_TTLE_NM, --직책명
NULL AS CAM_EMP_KIND_CD, 
NULL AS CAM_YRLY_SLRY_TYPE,   
NULL AS PRBTN_YN, 
NULL AS PRBTN_END_DT,  
NULL AS JOB_SCND_DEPT_ID,  
NULL AS JOB_SCND_JOB_GD_CD, 
NULL AS JOB_SCND_JOB_RANK_CD, 
NULL AS JOB_SCND_JOB_CD, 
NULL AS JOB_SCND_JOB_TTLE_CD, 
NULL AS LAYOFF_YN,
COALESCE(dire.DTL_NM, dore.DTL_NM, stre.DTL_NM, past.HR_CD_NM) AS CAM_CLASS_NM, --발령분류명 
case when mcls.MNG_DC = '1' then 'VolRetCl1'
     when mcls.MNG_DC = '2' then 'VolRetCl2'
     else 'EE9999' end RETRD_TYPE,
case when mcls.MNG_DC = '1' then '자발적'
     when mcls.MNG_DC = '2' then '비자발적'
    else '코드없음'
    end RETRD_TYPE_NM
from slv.system_a.cam_past past
LEFT JOIN (  
  SELECT A.EMP_NO, A.MCLS_CD, A.COMPANY_CD, A.MNG_DC, RETR_DT  
  FROM slv.system_a.HR_EMP_SDTL A  
  INNER JOIN slv.system_a.HR_EMP_MST B ON A.COMPANY_CD = B.COMPANY_CD AND A.EMP_NO = B.EMP_NO  
  INNER JOIN slv.system_a.HR_EMPINFO_DTL HED ON HED.COMPANY_CD = b.COMPANY_CD AND HED.EMP_NO = b.EMP_NO  
  WHERE A.COMPANY_CD IN ('1000','2000','3000')  
  AND (  
    (A.COMPANY_CD = '1000' AND MCLS_CD = 'HR002')  
    OR (A.COMPANY_CD = '2000' AND MCLS_CD = 'HR002')  
    OR (A.COMPANY_CD = '3000' AND MCLS_CD = 'HR002')  
  )  
  AND RETR_DT IS NOT NULL   ---자발/비자발 마스터
) mcls ON past.COMPANY_CD = mcls.COMPANY_CD AND past.EMP_NO = mcls.EMP_NO AND TO_CHAR(TO_TIMESTAMP(past.GNFD_DT , 'yyyyMMdd'), 'yyyyMMdd') = mcls.RETR_DT 
left join ref.hr.code_master dire on dire.df_cd = 'VolRetCl' AND dire.Value3 IS NOT NULL and dire.Value3 = mcls.MNG_DC  --공통코드 발령분류명 변경
left join ref.hr.code_master dore on dore.df_cd = 'VolRetCl' AND dore.Value5 IS NOT NULL and dore.Value5 = mcls.MNG_DC  --공통코드 발령분류명 변경
left join ref.hr.code_master stre on stre.df_cd = 'VolRetCl' AND stre.Value7 IS NOT NULL and stre.Value7 = mcls.MNG_DC  --공통코드 발령분류명 변경
""")

In [ ]:
DF.write.mode("overwrite").save("abfss://gld@storage_account.dfs.core.windows.net/f_hr_appoint_history");

In [ ]:
%%sql
USE gld.hr;
CREATE TABLE IF NOT EXISTS f_hr_appoint_history
USING DELTA
LOCATION 'abfss://gld@storage_account.dfs.core.windows.net/f_hr_appoint_history';